# 07 — Mean-variance optimization of a CSV portfolio

> **Universe build** (`ryan/universe`). Same model code as `ryan/sample`; the
> universe, tables and history come from `config.py` — profile set by the
> `BARRA_UNIVERSE` environment variable (default `r1000`). See the README.

Takes the same CSV as 06 and re-weights it:

$$\max_w \;\; \alpha^\top w - \tfrac{\lambda}{2}\, w^\top \Sigma w
\quad \text{s.t.} \quad \textstyle\sum_i w_i = 1,\;\; 0 \le w_i \le w_{\max}$$

with Σ from the factor model, never formed. The optimizer works in factor
space: $\Sigma w = X\,(F\,(X^\top w)) + \Delta w$.

### Expected returns (α)

- If the CSV has an **`expected_return`** column (annualized excess return),
  that is used.
- Otherwise α = ½ × historical mean **style** factor returns × exposures, using
  only factor returns up to the forecast date. This is an **illustration of the
  machinery, not a forecast.** 02 prints each style premium's t-statistic; on
  the sample the best was value at t = 2.1. For an honest risk-only answer, read
  the **minimum-variance** portfolio, which needs no α.

### Why the risk model's corrections matter here

A min-variance optimizer is the harshest consumer a covariance estimate has. It
goes looking for whichever direction Σ̂ understates most. Section 5 shows that
directly by optimizing under the raw sample F and re-pricing the result under
the corrected one. Section 6 checks it out of sample.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

# ryan/universe/config.py -- imported FIRST: it shadows the sample's config for
# every shared module and puts ryan/sample (where the model code lives) on the path.
import config
assert config.PROFILE != "sample", "wrong config on the path — run this notebook from ryan/universe"
print(config.describe())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import optimizer
import portfolio
import riskmath
import riskmodel

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN, AMBER = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a", "#eda100"
PTF_COLORS = {"current": MUTED, "min-variance": GREEN, "mean-variance": INK, "max α/vol": AMBER}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


# ---- inputs -----------------------------------------------------------------
PORTFOLIO_CSV = config.PORTFOLIO_CSV
AS_OF = None                          # None = latest forecast date
UNIVERSE = "portfolio"                # "portfolio": re-weight the CSV names; "universe": any covered name
LONG_ONLY = True
MAX_WEIGHT = config.MAX_WEIGHT        # per-name cap
RISK_AVERSION = config.RISK_AVERSION  # λ, annualized units
RUN_WALK_FORWARD = True               # section 6: re-optimize at every past date (~30 s)

In [ ]:
engine = db.get_engine()
db.check_connection(engine)
store = riskmodel.ModelStore.from_db(engine)
model = store.at(AS_OF)

holdings, _ = portfolio.load_portfolio(PORTFOLIO_CSV, store, model.date)
w_cur = holdings["weight"]
ids = (w_cur.index if UNIVERSE == "portfolio"
       else model.ids[model.meta["in_coverage"].reindex(model.ids).fillna(False).astype(bool).to_numpy()])
if MAX_WEIGHT * len(ids) < 1:
    raise ValueError(f"MAX_WEIGHT {MAX_WEIGHT} × {len(ids)} names < 100% — raise the cap or add names")
print(f"\noptimizing over {len(ids)} names ({UNIVERSE}), long-only={LONG_ONLY}, "
      f"max weight {MAX_WEIGHT:.0%}, λ = {RISK_AVERSION}")

---
## 1 — Expected returns

In [ ]:
if "expected_return" in holdings.columns and holdings["expected_return"].notna().all() and UNIVERSE == "portfolio":
    alpha = holdings["expected_return"].astype(float)
    alpha_src = "CSV expected_return column"
else:
    alpha = portfolio.default_alpha(model, store.factor_returns, config.ALPHA_PREMIA_SHRINK, ids)
    alpha_src = f"{config.ALPHA_PREMIA_SHRINK} × historical style premia × exposures (illustrative)"
print(f"α source: {alpha_src}")

styles = [f for f in model.factors if riskmodel._group(f) == "style"]
premia = store.factor_returns.loc[:model.date, styles].mean() * 12
display(premia.to_frame("historical mean %/yr").mul(100).round(2).T)

vols = model.asset_vols()
assets = pd.DataFrame({"alpha": alpha, "vol": vols["total_vol"].reindex(ids)}).join(model.meta[["ticker"]])
display(assets.sort_values("alpha", ascending=False).head(8).style.format({"alpha": "{:.2%}", "vol": "{:.1%}"}))

---
## 2 — Optimize

In [ ]:
w_mv = optimizer.optimize(model, ids, alpha=None, long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
w_opt = optimizer.optimize(model, ids, alpha=alpha, risk_aversion=RISK_AVERSION,
                           long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
front = optimizer.frontier(model, ids, alpha, n_points=30, long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
w_sharpe = front.loc[front["sharpe"].idxmax(), "weights"]

ports = {"current": w_cur.reindex(ids, fill_value=0.0), "min-variance": w_mv,
         "mean-variance": w_opt, "max α/vol": w_sharpe}
bench = model.market_weights(coverage_only=True)


def describe(w):
    d = model.decompose(w)
    er = float(alpha.reindex(w.index).fillna(0) @ w)
    return {
        "predicted vol": d["total_vol"], "expected return (α)": er, "α / vol": er / d["total_vol"],
        "factor share of var": d["pct_factor"],
        "beta to sample mkt": float(model.betas().reindex(w.index) @ w),
        "names held": int((w.abs() > 1e-4).sum()), "largest weight": float(w.max()),
        "turnover vs current (one-way)": 0.5 * float((w - ports["current"]).abs().sum()),
        **{f"{s} exposure": float(v) for s, v in model.exposures_of(w)[styles].items()},
    }


comp = pd.DataFrame({k: describe(v) for k, v in ports.items()})
display(comp.style.format("{:.3f}"))
print(f"min-variance cuts predicted vol from {comp.loc['predicted vol', 'current']:.2%} to "
      f"{comp.loc['predicted vol', 'min-variance']:.2%} "
      f"({1 - comp.loc['predicted vol', 'min-variance'] / comp.loc['predicted vol', 'current']:.0%} lower)")

---
## 3 — The efficient frontier

Swept from risk-neutral to minimum-variance. Individual names are shown for
scale. The frontier sits well to the left of them, and that gap is
diversification.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.scatter(assets["vol"] * 100, assets["alpha"] * 100, s=14, color=MUTED, alpha=.6, lw=0)
for _, r in assets.nlargest(4, "alpha").iterrows():
    ax.annotate(r["ticker"], (r["vol"] * 100, r["alpha"] * 100), fontsize=8, color=MUTED,
                xytext=(3, 2), textcoords="offset points")
ax.plot(front["vol"] * 100, front["expected_return"] * 100, color=INK, lw=2)
for name, w in ports.items():
    v, e = model.vol(w) * 100, float(alpha.reindex(w.index).fillna(0) @ w) * 100
    ax.scatter(v, e, s=90, color=PTF_COLORS[name], zorder=5, edgecolor="white", lw=1.5)
    ax.annotate(f"  {name}", (v, e), color=PTF_COLORS[name], fontsize=10, va="center", fontweight="medium")
ax.set(title=f"Efficient frontier at {model.date.date()} ({'long-only, ' if LONG_ONLY else ''}≤{MAX_WEIGHT:.0%} per name)",
       xlabel="predicted vol, %/yr", ylabel="expected return α, %/yr")
tidy(ax)
plt.tight_layout()
plt.show()

---
## 4 — The trades

In [ ]:
tbl = pd.DataFrame({k: v for k, v in ports.items()}).fillna(0.0)
tbl = tbl.join(model.meta[["ticker", "sector_name"]]).join(assets["alpha"])
tbl["trade → mean-variance"] = tbl["mean-variance"] - tbl["current"]
tbl = tbl[(tbl[list(ports)].abs() > 1e-4).any(axis=1)].sort_values("mean-variance", ascending=False)
display(tbl[["ticker", "sector_name", "alpha", *ports, "trade → mean-variance"]]
        .style.format({c: "{:.2%}" for c in ["alpha", *ports, "trade → mean-variance"]}))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
show = tbl.sort_values("current", ascending=False)
pos = np.arange(len(show))
width = .27
for j, name in enumerate(["current", "min-variance", "mean-variance"]):
    ax.bar(pos + (j - 1) * width, show[name] * 100, width * .95, color=PTF_COLORS[name], label=name)
ax.set_xticks(pos, show["ticker"], rotation=60, ha="right", fontsize=8)
ax.axhline(MAX_WEIGHT * 100, color=ACCENT, lw=1, ls="--")
ax.annotate(f"cap {MAX_WEIGHT:.0%}", (pos[-1], MAX_WEIGHT * 100), color=ACCENT, fontsize=8,
            xytext=(0, 4), textcoords="offset points", ha="right")
ax.set(ylabel="weight, %", title="Weights by portfolio")
ax.legend(frameon=False, fontsize=9)
tidy(ax)
plt.tight_layout()
plt.show()

---
## 5 — What the corrected covariance changes

The same min-variance problem, solved with the **raw** 60-month sample F (no
decay, no eigenfactor adjustment, no VRA), then re-priced under the final F.
The raw model believes its own answer is safer than the final model does. That
gap is the understated eigen-direction the optimizer found.

**How big the gap gets depends on how much freedom the optimizer has.** A
long-only book of ~20 names with a 10% cap can't lean far into any one
direction, so the difference here is a few percent. With `UNIVERSE = "universe"`,
or with shorts allowed, it grows. 05's unconstrained long-short test shows the
full effect.

In [ ]:
model_raw = store.at(model.date, cov_col="cov_raw")
w_mv_raw = optimizer.optimize(model_raw, ids, alpha=None, long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
gap = pd.DataFrame({
    "vol under raw F": [model_raw.vol(w_mv_raw), model_raw.vol(w_mv)],
    "vol under final F": [model.vol(w_mv_raw), model.vol(w_mv)],
}, index=["min-var built with raw F", "min-var built with final F"])
gap["final / raw"] = gap["vol under final F"] / gap["vol under raw F"]
display(gap.style.format({"vol under raw F": "{:.2%}", "vol under final F": "{:.2%}", "final / raw": "{:.3f}"}))
print(f"weight overlap between the two min-variance portfolios: "
      f"{np.minimum(w_mv_raw.reindex(ids, fill_value=0), w_mv.reindex(ids, fill_value=0)).sum():.0%}")

---
## 6 — Walk-forward: was the optimized risk real?

At every past forecast date, re-optimize over whichever of these names were
alive, using only that date's model. Hold for one month and record the
realized return. The current weights, renormalized over live names, are the
comparison. Every point is out of sample.

⚠️ Same caveat as 06: today's names run backwards are survivors. This tests
whether the **predicted** risk of the optimized portfolio was honest, and
whether min-variance actually delivered lower realized risk. It is not a
claim about returns.

In [ ]:
if RUN_WALK_FORWARD:
    R = store.realized_returns()
    nxt = dict(zip(store.dates[:-1], store.dates[1:]))
    rows = []
    for d in store.dates[:-1]:
        m = store.at(d)
        live = pd.Index(ids).intersection(m.ids)
        r1 = R.loc[nxt[d]].reindex(live)
        live = live[r1.notna().to_numpy()]
        if len(live) < 0.8 * len(ids) or MAX_WEIGHT * len(live) < 1:
            continue
        mr = store.at(d, cov_col="cov_raw")
        wc = w_cur.reindex(live).fillna(0)
        wc = wc / wc.sum() if wc.sum() > 0 else pd.Series(1 / len(live), live)
        wm = optimizer.optimize(m, live, long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
        wr = optimizer.optimize(mr, live, long_only=LONG_ONLY, max_weight=MAX_WEIGHT)
        for name, w_, mm in (("current", wc, m), ("min-var (final F)", wm, m), ("min-var (raw F)", wr, mr)):
            rows.append({"date": d, "portfolio": name, "pred_vol": mm.vol(w_),
                         "realized": float(w_ @ r1.reindex(w_.index))})
    WF = pd.DataFrame(rows)
    WF["z"] = WF["realized"] / (WF["pred_vol"] / np.sqrt(12))
    summ = WF.groupby("portfolio").agg(
        months=("z", "size"),
        mean_predicted_vol=("pred_vol", "mean"),
        realized_vol=("realized", lambda s: s.std() * np.sqrt(12)),
        bias_stat=("z", lambda s: riskmath.bias_stat(s)),
    )
    lo, hi = riskmath.bias_ci(int(summ["months"].iloc[0]))
    display(summ.style.format({"mean_predicted_vol": "{:.2%}", "realized_vol": "{:.2%}", "bias_stat": "{:.3f}"}))
    print(f"95% band for the bias statistic: [{lo:.3f}, {hi:.3f}]")
    print("Read across: realized vol says whether min-variance actually reduced risk; the bias")
    print("statistic says whether the model told the truth about how much.")

In [ ]:
if RUN_WALK_FORWARD:
    fig, ax = plt.subplots(figsize=(13, 3.6))
    colors = {"current": MUTED, "min-var (final F)": GREEN, "min-var (raw F)": ACCENT}
    ends = {}
    for name, g in WF.groupby("portfolio"):
        rv = g.set_index("date")["realized"].rolling(12).std().shift(-11) * np.sqrt(12) * 100
        rv = rv.dropna()
        ax.plot(rv.index, rv, color=colors[name], lw=2)
        ends[name] = (rv.index[-1], rv.iloc[-1])
    # Nudge end labels apart; the two min-var lines finish almost on top of each other.
    last = -np.inf
    for name in sorted(ends, key=lambda k: ends[k][1]):
        y = max(ends[name][1], last + 1.1)
        last = y
        ax.annotate(f" {name}", (ends[name][0], y), color=colors[name], fontsize=9, va="center")
    ax.set(title="Realized vol over the following 12 months, walk-forward", ylabel="%/yr")
    ax.margins(x=.14)
    tidy(ax)
    plt.tight_layout()
    plt.show()

---
## 7 — Save the optimized weights

Written next to the input CSV in the same format, so they can be fed straight
back into 06.

In [ ]:
out_dir = os.path.dirname(PORTFOLIO_CSV) or "."
stem = os.path.splitext(os.path.basename(PORTFOLIO_CSV))[0]
for name in ("min-variance", "mean-variance"):
    w_ = ports[name][ports[name] > 1e-4]
    df = pd.DataFrame({"id": w_.index, "ticker": model.meta.loc[w_.index, "ticker"].to_numpy(),
                       "weight": w_.round(6).to_numpy()})
    path = os.path.join(out_dir, f"{stem}_{name.replace('-', '_')}.csv")
    df.to_csv(path, index=False)
    print(f"💾 {path}  ({len(df)} names)")